# 02 - Validacao das Integras TXT

Objetivo: validar a ligacao entre os metadados JSON e os arquivos TXT no ZIP usando `SeqDocumento`, sem fazer ainda analise textual substantiva.

Esta etapa tambem pode montar uma amostra pequena com texto integral para alimentar o notebook seguinte.


## 1. Ambiente

No Colab, clone o repositorio antes de abrir/rodar o notebook, ou abra o notebook diretamente pelo GitHub. Os dados brutos devem ficar no Google Drive, fora do Git.

In [ ]:
# Rode esta celula no Colab se as dependencias ainda nao estiverem instaladas.
# !pip install -r requirements.txt

In [ ]:
from pathlib import Path
import json
import re
import zipfile

import pandas as pd
from bs4 import BeautifulSoup
from tqdm.auto import tqdm

In [ ]:
from pathlib import Path
from google.colab import drive

MOUNT_POINT = Path("/content/drive")

if (MOUNT_POINT / "MyDrive").exists():
    print("Google Drive já está montado.")
else:
    drive.mount(str(MOUNT_POINT), force_remount=True)

DRIVE_DATA = Path("/content/drive/MyDrive/Mestrado/2026/llms/data")
RAW_STJ = DRIVE_DATA / "raw" / "stj_integras"
INTERIM_DATA = DRIVE_DATA / "interim"
PROCESSED_DATA = DRIVE_DATA / "processed"
REPORTS_DATA = DRIVE_DATA / "reports" / "summaries"

for path in [RAW_STJ, INTERIM_DATA, PROCESSED_DATA, REPORTS_DATA]:
    path.mkdir(parents=True, exist_ok=True)

print("RAW_STJ:", RAW_STJ)
print("Arquivos encontrados:")
for file in sorted(RAW_STJ.iterdir()):
    print("-", file.name)


## 2. Inspecao dos arquivos brutos

Esperado em `data/raw/stj_integras/` no Drive:

- um JSON de metadados;
- um ZIP com textos integrais em TXT;
- um CSV com dicionario de dados.

In [ ]:
def inspect_raw_folder(raw_dir: str | Path) -> dict[str, list[Path]]:
    raw_dir = Path(raw_dir)
    files = sorted([p for p in raw_dir.iterdir() if p.is_file()])
    return {
        'json': [p for p in files if p.suffix.lower() == '.json'],
        'zip': [p for p in files if p.suffix.lower() == '.zip'],
        'csv': [p for p in files if p.suffix.lower() == '.csv'],
        'other': [p for p in files if p.suffix.lower() not in {'.json', '.zip', '.csv'}],
    }

raw_files = inspect_raw_folder(RAW_STJ)
for kind, files in raw_files.items():
    print(kind.upper())
    for file in files:
        print(' ', file.name)

In [ ]:
# Ajuste manualmente caso a pasta tenha mais de um arquivo do mesmo tipo.
METADATA_PATH = raw_files["json"][0] if raw_files["json"] else None
ZIP_PATH = raw_files["zip"][0] if raw_files["zip"] else None
DICTIONARY_PATH = raw_files["csv"][0] if raw_files["csv"] else None

if METADATA_PATH is None:
    raise FileNotFoundError(f"Nenhum arquivo .json encontrado em {RAW_STJ}")
if ZIP_PATH is None:
    raise FileNotFoundError(f"Nenhum arquivo .zip encontrado em {RAW_STJ}")

print("METADATA_PATH:", METADATA_PATH)
print("ZIP_PATH:", ZIP_PATH)
print("DICTIONARY_PATH:", DICTIONARY_PATH)


## 3. Leitura dos metadados

In [ ]:
def load_metadata(path: str | Path) -> pd.DataFrame:
    path = Path(path)
    with path.open('r', encoding='utf-8') as f:
        data = json.load(f)

    if isinstance(data, list):
        df = pd.DataFrame(data)
    elif isinstance(data, dict):
        list_values = [v for v in data.values() if isinstance(v, list)]
        if len(list_values) == 1:
            df = pd.DataFrame(list_values[0])
        else:
            df = pd.json_normalize(data)
    else:
        raise ValueError(f'Formato JSON inesperado: {type(data)}')

    print(f'Metadados: {df.shape[0]:,} linhas x {df.shape[1]:,} colunas')
    return df

metadata_df = load_metadata(METADATA_PATH)
metadata_df.head()

In [ ]:
metadata_df.info()

In [ ]:
metadata_df.columns.tolist()

## 4. Validacao do ZIP e da chave `SeqDocumento`

A validacao compara os `SeqDocumento` presentes nos metadados com os nomes dos arquivos TXT dentro do ZIP.


In [ ]:
def list_zip_txt_files(zip_path: str | Path) -> list[str]:
    zip_path = Path(zip_path)
    with zipfile.ZipFile(zip_path) as zf:
        return sorted([
            name for name in zf.namelist()
            if not name.endswith('/') and name.lower().endswith('.txt')
        ])

txt_files = list_zip_txt_files(ZIP_PATH)
print(f'TXT no ZIP: {len(txt_files):,}')
txt_files[:10]

## 6. Validacao da chave `SeqDocumento`

In [ ]:
def normalize_seq_documento(value) -> str | None:
    if pd.isna(value):
        return None
    text = str(value).strip()
    if text.endswith('.0'):
        text = text[:-2]
    return text

def txt_stem(name: str) -> str:
    return Path(name).stem.strip()

def validate_seqdocumento_link(metadata_df: pd.DataFrame, txt_files: list[str]) -> dict:
    if 'SeqDocumento' not in metadata_df.columns:
        raise KeyError('Coluna SeqDocumento nao encontrada nos metadados')

    metadata_keys = set(
        metadata_df['SeqDocumento']
        .map(normalize_seq_documento)
        .dropna()
    )
    txt_keys = {txt_stem(name) for name in txt_files}

    matched = metadata_keys & txt_keys
    metadata_without_txt = metadata_keys - txt_keys
    txt_without_metadata = txt_keys - metadata_keys

    return {
        'total_metadata_rows': len(metadata_df),
        'total_metadata_keys': len(metadata_keys),
        'total_txt_files': len(txt_files),
        'matched_keys': len(matched),
        'metadata_without_txt': len(metadata_without_txt),
        'txt_without_metadata': len(txt_without_metadata),
        'metadata_without_txt_examples': sorted(metadata_without_txt)[:20],
        'txt_without_metadata_examples': sorted(txt_without_metadata)[:20],
    }

validation_report = validate_seqdocumento_link(metadata_df, txt_files)
validation_report

In [ ]:
validation_path = REPORTS_DATA / 'stj_integras_key_validation.json'
validation_path.write_text(json.dumps(validation_report, ensure_ascii=False, indent=2), encoding='utf-8')
validation_path

## 7. Limpeza textual basica

In [ ]:
def clean_legal_text(text: str) -> str:
    if text is None or pd.isna(text):
        return ''
    text = str(text)
    text = re.sub(r'<\s*br\s*/?\s*>', '\n', text, flags=re.IGNORECASE)
    text = BeautifulSoup(text, 'html.parser').get_text(' ')
    text = re.sub(r'\r\n?', '\n', text)
    text = re.sub(r'[ \t]+', ' ', text)
    text = re.sub(r'\n\s*\n+', '\n\n', text)
    return text.strip()

clean_legal_text('Linha 1<br>Linha 2<br/> <b>Texto</b> juridico.')

## 8. Construcao de corpus amostral

In [ ]:
EXPECTED_COLUMNS = [
    'SeqDocumento',
    'dataPublicacao',
    'tipoDocumento',
    'processo',
    'ministro',
    'NM_MINISTRO',
    'teor',
    'descricaoMonocratica',
    'assuntos',
]

def read_txt_from_zip(zf: zipfile.ZipFile, file_name: str) -> str:
    raw = zf.read(file_name)
    for encoding in ['utf-8', 'latin-1', 'cp1252']:
        try:
            return raw.decode(encoding)
        except UnicodeDecodeError:
            continue
    return raw.decode('utf-8', errors='replace')

def build_sample_corpus(metadata_df: pd.DataFrame, zip_path: str | Path, n: int = 50, random_state: int = 42) -> pd.DataFrame:
    zip_path = Path(zip_path)
    df = metadata_df.copy()
    df['_seq_key'] = df['SeqDocumento'].map(normalize_seq_documento)

    with zipfile.ZipFile(zip_path) as zf:
        txt_files = list_zip_txt_files(zip_path)
        txt_by_key = {txt_stem(name): name for name in txt_files}
        available = df[df['_seq_key'].isin(txt_by_key)].copy()

        sample_n = min(n, len(available))
        sample = available.sample(n=sample_n, random_state=random_state) if sample_n else available

        rows = []
        for _, row in tqdm(sample.iterrows(), total=len(sample)):
            key = row['_seq_key']
            texto_original = read_txt_from_zip(zf, txt_by_key[key])
            item = {col: row[col] if col in row.index else None for col in EXPECTED_COLUMNS}
            if not item.get('ministro') and item.get('NM_MINISTRO'):
                item['ministro'] = item['NM_MINISTRO']
            item['texto_original'] = texto_original
            item['texto_limpo'] = clean_legal_text(texto_original)
            rows.append(item)

    return pd.DataFrame(rows)

sample_corpus = build_sample_corpus(metadata_df, ZIP_PATH, n=50)
sample_corpus.head()

In [ ]:
sample_corpus.assign(
    chars_original=sample_corpus['texto_original'].str.len(),
    chars_limpo=sample_corpus['texto_limpo'].str.len(),
)[['SeqDocumento', 'tipoDocumento', 'ministro', 'teor', 'chars_original', 'chars_limpo']].head(10)

In [ ]:
sample_parquet = PROCESSED_DATA / 'stj_integras_sample.parquet'
sample_csv = PROCESSED_DATA / 'stj_integras_sample.csv'

sample_corpus.to_parquet(sample_parquet, index=False)
sample_corpus.to_csv(sample_csv, index=False)

print('Salvo em:')
print('-', sample_parquet)
print('-', sample_csv)

## 8. Proximos passos

- Conferir `stj_integras_key_validation.json`.
- Se a cobertura da chave estiver boa, usar a amostra salva em `processed/` no notebook `03_analise_textual_inicial.ipynb`.
- Se a cobertura estiver baixa, verificar se o JSON e o ZIP sao do mesmo pacote/data.
